# Grupo 4 — Random Forest para temperatura em Jena

Previsão causal da temperatura média da próxima hora. Execute as células do
início ao fim para regenerar resultados sob o protocolo comum do grupo 4.


### Informações do Grupo

##### Grupo: 4

Base: `jena_climate_2009_2016.csv`

Previsão da temperatura do ar na base Jena Climate.

- Granularidade: `horária` (agregação da base original de 10 minutos)
- Variável-alvo: `T (degC)`
- Random State (seed): `42`
- Treino-teste: `80% / 20%`, respeitando a ordem temporal

## Importações e Configurações Padrão 

In [ ]:
from pathlib import Path
import sys
import time

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestRegressor
from sklearn.inspection import permutation_importance
from sklearn.metrics import mean_absolute_error, mean_squared_error
from sklearn.model_selection import ParameterGrid, TimeSeriesSplit
from statsmodels.graphics.tsaplots import plot_acf
from statsmodels.stats.diagnostic import acorr_ljungbox



import sys

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "validacao_bases.py").is_file():
    PROJECT_ROOT = PROJECT_ROOT.parent
if not (PROJECT_ROOT / "validacao_bases.py").is_file():
    raise FileNotFoundError("Não foi possível localizar a raiz do projeto.")
sys.path.insert(0, str(PROJECT_ROOT))

from validacao_bases import relatorios_como_dataframe, validar_base
from funcoes_series_temporais import analisar_componente_residual, calcular_forca_sazonalidade


In [ ]:
DATA_FILENAME = "jena_climate_2009_2016.csv"
DATA_PATH = PROJECT_ROOT / "grupo4" / DATA_FILENAME
DATE_COLUMN = "Date Time"
TARGET_COLUMN = "T (degC)"
FREQUENCY = "1h"
TRAIN_RATIO = 0.80
SEED = 42
HORIZON = 1
SEASONAL_PERIOD = 24
FFILL_LIMIT_HOURS = 6
BASE_EXOG_COLUMNS = ["p (mbar)", "rh (%)", "wv (m/s)", "wd (deg)"]
MODEL_EXOG_COLUMNS = ["p (mbar)", "rh (%)", "wv (m/s)", "wd_sin", "wd_cos"]
N_JOBS = -1


## Configuração do Random Forest

In [ ]:
# Configuração do Random Forest para a mesma previsão de uma hora da MLP.
TARGET_LAGS = [1, 2, 3, 6, 12, 24, 48, 168]
EXOG_COLUMNS = MODEL_EXOG_COLUMNS
EXOG_LAGS = {coluna: [1, 24] for coluna in EXOG_COLUMNS}
ROLLING_WINDOWS = [6, 24, 168]
PARAM_GRID = {
    "n_estimators": [100, 200],
    "max_depth": [10, None],
    "min_samples_split": [2, 5],
    "min_samples_leaf": [1, 2, 5],
    "max_features": ["sqrt", 1.0],
}
N_JOBS = -1
N_SPLITS_VALIDACAO = 3
HORAS_VALIDACAO = 24 * 28
SAIDA = PROJECT_ROOT / "resultados_random_forest"
SAIDA.mkdir(exist_ok=True)


# 1. Preparação da base

## T01 — Sanitização, valores ausentes e decisões de limpeza

A base original tem registros de 10 minutos e é agregada para a granularidade horária. O marcador `-9999` é convertido para nulo antes da agregação. O alvo não é imputado. Para evitar vazamento temporal, somente as exógenas usam a última observação disponível (`ffill`), limitada a 6 horas; valores posteriores nunca são usados. Registros ainda incompletos são removidos na construção das features.

A validação sanitária é somente leitura e registra duplicatas, timestamps ausentes e nulos da base original. Essas ocorrências não são corrigidas silenciosamente: as decisões de tratamento são explícitas no código da etapa T01.

In [ ]:
relatorio = validar_base("clima", raiz=PROJECT_ROOT)
display(relatorios_como_dataframe({"clima": relatorio}))

# T01 — limpeza e agregação horária padronizadas no grupo 4.
df_bruto = pd.read_csv(DATA_PATH)
df_bruto[DATE_COLUMN] = pd.to_datetime(
    df_bruto[DATE_COLUMN], format="%d.%m.%Y %H:%M:%S", errors="coerce"
)
n_datas_invalidas = int(df_bruto[DATE_COLUMN].isna().sum())
n_duplicadas = int(df_bruto[DATE_COLUMN].duplicated().sum())
df_bruto = (
    df_bruto.dropna(subset=[DATE_COLUMN])
    .sort_values(DATE_COLUMN)
    .drop_duplicates(subset=[DATE_COLUMN], keep="first")
    .set_index(DATE_COLUMN)
)

# -9999 é ausência, portanto deve ser removido antes da média horária.
df_numerico = df_bruto.select_dtypes(include="number").replace(-9999, np.nan)
df = df_numerico.resample(FREQUENCY).mean()

# O alvo nunca é imputado. Somente exógenas recebem preenchimento causal.
alvo_ausente = df[TARGET_COLUMN].isna().copy()
for coluna in BASE_EXOG_COLUMNS:
    df[coluna] = df[coluna].ffill(limit=FFILL_LIMIT_HOURS)
df.loc[alvo_ausente, TARGET_COLUMN] = np.nan

# Direção é circular: 0 e 360 graus representam a mesma direção.
df["wd_sin"] = np.sin(np.deg2rad(df["wd (deg)"]))
df["wd_cos"] = np.cos(np.deg2rad(df["wd (deg)"]))

if not df.index.is_monotonic_increasing or not df.index.is_unique:
    raise ValueError("O índice horário deve ser crescente e único.")
if (df["wv (m/s)"].dropna() < 0).any():
    raise ValueError("Há velocidade do vento negativa depois da limpeza.")

# O corte é definido na série horária completa, antes dos lags dos modelos.
posicao_corte = int(len(df) * TRAIN_RATIO)
FIM_TREINO = df.index[posicao_corte - 1]
INICIO_TESTE = df.index[posicao_corte]
diagnostico_base = pd.DataFrame([{
    "linhas_horarias": len(df),
    "datas_invalidas": n_datas_invalidas,
    "duplicatas_removidas": n_duplicadas,
    "horas_sem_alvo": int(df[TARGET_COLUMN].isna().sum()),
    "frequencia": FREQUENCY,
    "ffill_limite_horas": FFILL_LIMIT_HOURS,
    "fim_treino": FIM_TREINO,
    "inicio_teste": INICIO_TESTE,
}])
display(diagnostico_base)
print(f"Período: {df.index.min()} a {df.index.max()}")
print("O alvo não foi imputado; exógenas usam ffill causal limitado.")


## T02 — Dicionário de dados e disponibilidade temporal

Criação do dicionário de dados e análise de disponibilidade temporal das variáveis exógenas para prevenir data leakage.

In [ ]:
# Dicionário e disponibilidade temporal das variáveis
DICIONARIO = pd.DataFrame([
    {"variavel": coluna, "tipo": str(df[coluna].dtype),
     "disponibilidade": "observada; usada apenas com defasagens",
     "min": df[coluna].min(), "max": df[coluna].max(), "media": df[coluna].mean()}
    for coluna in [TARGET_COLUMN] + EXOG_COLUMNS
])
display(DICIONARIO.round(3))

## T03 — Análise exploratória visual

Análise visual do alvo e das variáveis exógenas, diagnóstico de outliers e registro das decisões de limpeza.

In [ ]:
# Análise exploratória visual
fig, axes = plt.subplots(len(EXOG_COLUMNS) + 1, 1, figsize=(14, 10), sharex=True, squeeze=False)
axes = axes[:, 0]
axes[0].plot(df.index, df[TARGET_COLUMN], color="#1a3a6e", label="Temperatura")
axes[0].set_title("Jena Climate — alvo e variáveis exógenas")
axes[0].set_ylabel("Temperatura (°C)")
axes[0].legend(); axes[0].grid(alpha=0.3)
for ax, coluna in zip(axes[1:], EXOG_COLUMNS):
    ax.plot(df.index, df[coluna], linewidth=0.7, label=coluna)
    ax.set_title(coluna); ax.set_ylabel(coluna); ax.legend(); ax.grid(alpha=0.3)
axes[-1].set_xlabel("Data e hora")
plt.tight_layout(); plt.show()

# 2. Sazonalidade

## T04 — Decomposição STL

A STL é calculada com `period=24` para diagnosticar somente o ciclo diário na série horária. A força de tendência deve ser interpretada com cautela: ela pode incorporar variações de prazo mais longo, inclusive a sazonalidade anual, que não é separada por esta decomposição. A sazonalidade anual é representada separadamente pelas features cíclicas de dia do ano.

In [ ]:
# Decomposição STL com período diário
stl_resultado = calcular_forca_sazonalidade(df[TARGET_COLUMN], periodo=SEASONAL_PERIOD)
decomposicao = stl_resultado["decomposicao"]
print(f"Força da sazonalidade diária: {stl_resultado['forca_sazonalidade']:.4f}")
print(
    "Força de tendência (pode incluir variação anual): "
    f"{stl_resultado['forca_tendencia']:.4f}"
)
fig = decomposicao.plot(); fig.set_size_inches(14, 8)
fig.suptitle("Decomposição STL — ciclo diário da temperatura", y=1.02)
plt.tight_layout(); plt.show()

## T05 — Força da sazonalidade e resíduos

A força sazonal indica quanto o componente diário reduz a variância residual. O componente residual é avaliado por média, dispersão e Ljung–Box. A autocorrelação residual indica dependência temporal ainda não explicada pelo modelo e deve ser interpretada em conjunto com o horizonte fixo de uma hora.

In [ ]:
# Força sazonal e componente residual
analise_residuos_stl = analisar_componente_residual(decomposicao, lags=24)
print(f"Média do residual: {analise_residuos_stl['media']:.6f}")
print(f"Desvio-padrão do residual: {analise_residuos_stl['desvio_padrao']:.6f}")
display(analise_residuos_stl["autocorrelacao_residual"])


# 3. Features e validação

## T06 — Pipeline de features

Construção do pipeline de features com lags do alvo e das exógenas, janelas móveis, variáveis de calendário e encoding cíclico.

In [ ]:
def construir_features_temporais(
    dados, alvo, *, lags_alvo, lags_exogenas, janelas_moveis
):
    """Cria features causais; o índice da linha representa a hora prevista."""
    features = pd.DataFrame(index=dados.index)
    for lag in sorted(set(lags_alvo)):
        if lag < HORIZON:
            raise ValueError(f"Lag do alvo deve ser >= {HORIZON}: {lag}")
        features[f"target_lag_{lag}"] = dados[alvo].shift(lag)
    for coluna, lags in lags_exogenas.items():
        for lag in sorted(set(lags)):
            if lag < HORIZON:
                raise ValueError(f"Lag da exógena deve ser >= {HORIZON}: {lag}")
            features[f"{coluna}_lag_{lag}"] = dados[coluna].shift(lag)
    passado = dados[alvo].shift(HORIZON)
    for janela in sorted(set(janelas_moveis)):
        features[f"target_roll_mean_{janela}"] = passado.rolling(janela).mean()
        features[f"target_roll_std_{janela}"] = passado.rolling(janela).std()
    features["hora_sin"] = np.sin(2 * np.pi * dados.index.hour / 24)
    features["hora_cos"] = np.cos(2 * np.pi * dados.index.hour / 24)
    features["dia_semana_sin"] = np.sin(2 * np.pi * dados.index.dayofweek / 7)
    features["dia_semana_cos"] = np.cos(2 * np.pi * dados.index.dayofweek / 7)
    features["dia_ano_sin"] = np.sin(2 * np.pi * dados.index.dayofyear / 365.25)
    features["dia_ano_cos"] = np.cos(2 * np.pi * dados.index.dayofyear / 365.25)
    return pd.concat([dados[alvo].rename("target"), features], axis=1).dropna()


model_data = construir_features_temporais(
    df, TARGET_COLUMN,
    lags_alvo=TARGET_LAGS,
    lags_exogenas=EXOG_LAGS,
    janelas_moveis=ROLLING_WINDOWS,
)
print(f"Features criadas: {model_data.shape[1] - 1} | observações utilizáveis: {len(model_data):,}")


## T07 — Escalonamento por janela

O Random Forest não exige escalonamento: as árvores usam pontos de corte e são invariantes à escala das variáveis. A separação temporal permanece necessária para evitar que informações do futuro participem do treinamento.

In [ ]:
# T07 — Random Forest não precisa de escalonamento; apenas do corte temporal.

train = model_data.loc[model_data.index < INICIO_TESTE]
test = model_data.loc[model_data.index >= INICIO_TESTE]
if train.empty or test.empty or train.index.max() >= test.index.min():
    raise ValueError("Divisão temporal inválida.")
y_train, y_test = train["target"], test["target"]
X_train, X_test = train.drop(columns="target"), test.drop(columns="target")

print(f"Treino: {len(train):,} | teste: {len(test):,}")
print(f"Corte comum: {FIM_TREINO} -> {INICIO_TESTE}")


## T08 — Validação walk-forward

Implementação do protocolo walk-forward com janelas, origens de teste fixas e horizonte de previsão unificado.

In [ ]:
# Cada linha do teste é uma origem de previsão de uma hora.
origens_teste = X_test.index
print(f'Origens de teste: {len(origens_teste):,} | horizonte: {HORIZON} hora')
print('As features de cada origem contêm somente observações até t-1.')


# 4. Modelagem Random Forest

## T11 — Otimização do Random Forest

Otimização de n_estimators, max_depth, min_samples_split, min_samples_leaf e max_features usando somente dados de desenvolvimento. A validação usa três blocos temporais do treino, com 28 dias cada; os hiperparâmetros selecionados são congelados antes da avaliação de uma hora fora da amostra.


In [ ]:
# Grid temporal: cada candidato é avaliado em três blocos futuros do treino.
inicio_grid = time.perf_counter()
parametros = list(ParameterGrid(PARAM_GRID))
cv = TimeSeriesSplit(
    n_splits=N_SPLITS_VALIDACAO,
    test_size=HORAS_VALIDACAO,
)
resultados_grid = []
for grid_index, params in enumerate(parametros):
    erros = []
    for indices_ajuste, indices_validacao in cv.split(X_train):
        modelo_grid = RandomForestRegressor(
            **params, random_state=SEED, n_jobs=N_JOBS
        )
        modelo_grid.fit(X_train.iloc[indices_ajuste], y_train.iloc[indices_ajuste])
        previsto = modelo_grid.predict(X_train.iloc[indices_validacao])
        erros.extend(y_train.iloc[indices_validacao].to_numpy() - previsto)
    resultados_grid.append({
        "grid_index": grid_index,
        **params,
        "MAE_validacao": np.abs(erros).mean(),
        "RMSE_validacao": np.sqrt(np.mean(np.square(erros))),
        "n_splits_validacao": N_SPLITS_VALIDACAO,
    })
    print(f"{grid_index + 1}/{len(parametros)} combinações testadas")

df_grid = pd.DataFrame(resultados_grid).sort_values("MAE_validacao").reset_index(drop=True)
display(df_grid.head(10))
melhor_indice = int(df_grid.loc[0, "grid_index"])
BEST_PARAMS = parametros[melhor_indice].copy()
print("Hiperparâmetros congelados:", BEST_PARAMS)
tempo_grid = time.perf_counter() - inicio_grid
df_grid.to_csv(SAIDA / "grid_random_forest_grupo4.csv", index=False)


## T12 — Random Forest walk-forward

Treinamento e execução walk-forward do Random Forest com hiperparâmetros congelados, gerando previsões e resíduos.

In [ ]:
# Modelo congelado no treino; previsões móveis de uma hora no teste.
inicio_walk_forward = time.perf_counter()
modelo_final = RandomForestRegressor(
    **BEST_PARAMS, random_state=SEED, n_jobs=N_JOBS
)
modelo_final.fit(X_train, y_train)
previsto = modelo_final.predict(X_test)
wf = pd.DataFrame({
    "modelo": "Random Forest",
    "origem": X_test.index - pd.Timedelta(hours=1),
    "data": X_test.index,
    "passo": 1,
    "real": y_test.to_numpy(),
    "previsto": previsto,
    "residuo": y_test.to_numpy() - previsto,
})
tempo_walk_forward = time.perf_counter() - inicio_walk_forward
print(f"Previsões de uma hora registradas: {len(wf):,}")
wf.to_csv(SAIDA / "previsoes_random_forest_grupo4.csv", index=False)


# 5. Avaliação

## T16 — Comparação por MAE

Consolidação do MAE fora da amostra por base e modelo, incluindo número de vitórias e posição média.

In [ ]:
# Métricas fora da amostra
metricas = pd.DataFrame([{
    "base": "clima",
    "modelo": "Random Forest",
    "MAE": wf["residuo"].abs().mean(),
    "RMSE": np.sqrt((wf["residuo"] ** 2).mean()),
    "ME_vies": wf["residuo"].mean(),
    "MaxAE": wf["residuo"].abs().max(),
    "tempo_grid_min": tempo_grid / 60,
    "tempo_walk_forward_min": tempo_walk_forward / 60,
}])
display(metricas.round(4))
print("MAE no horizonte de uma hora:")
mae_por_passo = wf.groupby("passo")["residuo"].apply(lambda s: s.abs().mean()).round(4)
display(mae_por_passo)
metricas.to_csv(SAIDA / "metricas_random_forest_grupo4.csv", index=False)
mae_por_passo.rename("MAE").to_csv(SAIDA / "mae_por_passo_random_forest_grupo4.csv")

## T17 — Diagnóstico dos resíduos

Análise gráfica e estatística dos resíduos com série temporal dos erros, ACF, teste de Ljung-Box e tabela de p-valores.

In [ ]:
# Diagnóstico dos resíduos fora da amostra para o horizonte de uma hora.
residuos = wf.sort_values("data").set_index("data")["residuo"]
fig, axes = plt.subplots(3, 1, figsize=(14, 11))
axes[0].plot(residuos.index, residuos, label="Resíduo — 1 hora")
axes[0].axhline(0, color="black", linestyle="--", label="Erro zero")
axes[0].set_title("Resíduos fora da amostra — Random Forest")
axes[0].legend(); axes[0].grid(alpha=0.3)
axes[1].hist(residuos, bins=30, alpha=0.75, label="Resíduos")
axes[1].legend(); axes[1].grid(alpha=0.3)
plot_acf(residuos, lags=48, ax=axes[2], title="ACF dos resíduos — 1 hora")
plt.tight_layout(); plt.show()

ljung_box = acorr_ljungbox(residuos, lags=range(1, 25), return_df=True)
display(ljung_box)
print("ME > 0 indica subestimação; ME < 0 indica superestimação.")
ljung_box.to_csv(SAIDA / "ljung_box_random_forest_grupo4.csv")


## T18 — Importância das features

Cálculo e interpretação da importância das features: permutation importance para modelos de tabela e coeficientes para o SARIMAX.

In [ ]:
# Importância nativa do modelo final; associativa, não causal.
importancias = pd.DataFrame({
    "feature": X_train.columns,
    "importancia": modelo_final.feature_importances_,
}).sort_values("importancia", ascending=False)
display(importancias.head(20))
top = importancias.head(15).sort_values("importancia")
fig, ax = plt.subplots(figsize=(10, 7))
ax.barh(top["feature"], top["importancia"], label="Importância nativa")
ax.set_title("Importância das features — Random Forest")
ax.set_xlabel("Redução média normalizada de impureza")
ax.legend(); ax.grid(axis="x", alpha=0.3)
plt.tight_layout(); plt.show()
importancias.to_csv(SAIDA / "importancia_random_forest_grupo4.csv", index=False)
